# Phase 4: p_success Logistic Regression Model

This notebook runs the Phase 4 implementation and inspects fitted coefficients, validation checks, subflow offsets, and calibration outputs.

In [ ]:
from pathlib import Path
import json
import pandas as pd

root = Path.cwd()
if (root / 'Simulation_4').exists():
    REPO_ROOT = root
else:
    REPO_ROOT = root.parent.parent

PHASE4_SCRIPT = REPO_ROOT / 'Simulation_4' / 'scripts' / 'phase 4' / 'phase4_psuccess_model.py'
MODEL_JSON = REPO_ROOT / 'Simulation_4' / 'artifacts' / 'phase 4' / 'psuccess_model.json'
CALIB_CSV = REPO_ROOT / 'Simulation_4' / 'artifacts' / 'phase 4' / 'psuccess_calibration.csv'
PHASE4_SCRIPT, MODEL_JSON, CALIB_CSV

In [ ]:
import subprocess

cmd = ['python', str(PHASE4_SCRIPT)]
run = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError(f'Phase 4 script failed with code {run.returncode}')

In [ ]:
with open(MODEL_JSON, 'r', encoding='utf-8') as f:
    payload = json.load(f)

print('Model notes')
display(pd.DataFrame([payload['model_notes']]))

print('Coefficients')
display(pd.DataFrame([payload['coefficients']]))

print('Validation checks')
display(pd.DataFrame([payload['validation']]))

In [ ]:
offsets = pd.DataFrame(list(payload['subflow_offsets'].items()), columns=['subflow', 'alpha_subflow'])
display(offsets.sort_values('alpha_subflow', ascending=False).head(5))
display(offsets.sort_values('alpha_subflow', ascending=True).head(5))

In [ ]:
cal = pd.read_csv(CALIB_CSV)
display(cal)

import matplotlib.pyplot as plt

plot = cal.copy()
plot['bin_center'] = [0.1, 0.3, 0.5, 0.7, 0.9]
plt.figure(figsize=(7, 4.5))
plt.plot(plot['bin_center'], plot['empirical_success_rate'], marker='o', label='empirical')
plt.plot(plot['bin_center'], plot['mean_predicted_psuccess'], marker='s', label='predicted')
plt.title('Phase 4 Calibration by info_proxy Bin')
plt.xlabel('info_proxy bin center')
plt.ylabel('success probability')
plt.ylim(0, 1)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()